In [2]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMRegressor
from sklift.models import SoloModel

import sys
sys.path.append("..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

# Tắt bớt log của Optuna để console không bị rối khi chạy nhiều seed
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu (Chỉ tải 1 lần ở ngoài vòng lặp)
print("Loading data...")
train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'spend'
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# 2. Khởi tạo danh sách các seed và list để lưu kết quả
seeds = [412312, 42, 1874, 902745, 1]
results = []

# 3. Chạy vòng lặp qua từng Seed
for SEED in seeds:
    print(f"\n{'='*50}")
    print(f"🚀 BẮT ĐẦU CHẠY VỚI SEED: {SEED}")
    print(f"{'='*50}")
    
    # Cố định seed cho môi trường
    seed_everything(SEED)

    # Hàm Objective định nghĩa bên trong để bắt được biến SEED hiện tại
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'num_leaves': trial.suggest_int('num_leaves', 10, 150),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'random_state': SEED, # Sử dụng current SEED
            'verbose': -1
        }
        
        base_model = LGBMRegressor(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        return score

    # Chạy Optuna
    print("🔃 Đang chạy Optuna Tuning...")
    fixed_sampler = TPESampler(seed=SEED)
    study = optuna.create_study(direction="maximize", study_name=f"S_Learner_LGBM_Tuning_{SEED}", sampler=fixed_sampler)
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    val_best_auqc = study.best_value
    best_params = study.best_params
    print(f"✅ Tuning hoàn tất! Best Val AUQC: {val_best_auqc:.4f}")

    # Huấn luyện mô hình Final
    print("🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...")
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['verbose'] = -1

    final_base_model = LGBMRegressor(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)

    # Dự đoán trên Test
    uplift_pred_test = final_s_learner.predict(X_test)

    # Đánh giá (Tắt plot=False để không in quá nhiều biểu đồ)
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    print(f"   Test AUUC: {auuc_score:.3f}")
    print(f"   Test AUQC: {auqc_score:.3f}")
    print(f"   Test Lift: {lift_score:.3f}")
    print(f"   Test KRCC: {krcc_score:.3f}")

    # Lưu kết quả của seed này vào dictionary
    run_result = {
        'seed': SEED,
        'val_best_auqc': val_best_auqc,
        'test_auuc': auuc_score,
        'test_auqc': auqc_score,
        'test_lift': lift_score,
        'test_krcc': krcc_score
    }
    # Gộp thêm các best parameters vào dict
    run_result.update(best_params)
    results.append(run_result)

# 4. Xuất kết quả ra file CSV
print("\n" + "="*50)
print("💾 ĐANG LƯU KẾT QUẢ VÀO FILE CSV...")
results_df = pd.DataFrame(results)
csv_filename = "s_learner_revenue_results.csv"
results_df.to_csv(csv_filename, index=False)
print(f"✅ Đã lưu thành công tại: {csv_filename}")

# Hiển thị thử bảng kết quả
display(results_df)

Loading data...

🚀 BẮT ĐẦU CHẠY VỚI SEED: 412312
Locked random seed: 412312
🔃 Đang chạy Optuna Tuning...


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 43. Best value: 0.686062: 100%|██████████| 50/50 [00:18<00:00,  2.66it/s]


✅ Tuning hoàn tất! Best Val AUQC: 0.6861
🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...
   Test AUUC: 0.545
   Test AUQC: 0.542
   Test Lift: 0.431
   Test KRCC: 0.028

🚀 BẮT ĐẦU CHẠY VỚI SEED: 42
Locked random seed: 42
🔃 Đang chạy Optuna Tuning...


Best trial: 12. Best value: 0.625674: 100%|██████████| 50/50 [00:35<00:00,  1.43it/s]


...

Best trial: 4. Best value: 0.574861: 100%|██████████| 50/50 [00:21<00:00,  2.34it/s]


✅ Tuning hoàn tất! Best Val AUQC: 0.5749
🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...
   Test AUUC: 0.458
   Test AUQC: 0.456
   Test Lift: 0.713
   Test KRCC: 0.010

🚀 BẮT ĐẦU CHẠY VỚI SEED: 1
Locked random seed: 1
🔃 Đang chạy Optuna Tuning...


Best trial: 33. Best value: 0.735319: 100%|██████████| 50/50 [00:31<00:00,  1.58it/s]


✅ Tuning hoàn tất! Best Val AUQC: 0.7353
🔃 Huấn luyện mô hình Final & Đánh giá trên tập TEST...
   Test AUUC: 0.697
   Test AUQC: 0.706
   Test Lift: 1.024
   Test KRCC: 0.132

💾 ĐANG LƯU KẾT QUẢ VÀO FILE CSV...
✅ Đã lưu thành công tại: s_learner_revenue_results.csv


,seed,val_best_auqc,test_auuc,test_auqc,test_lift,test_krcc,n_estimators,learning_rate,max_depth,num_leaves,min_child_samples,subsample,colsample_bytree
0,412312,0.686062,0.544639,0.542324,0.431287,0.027728,110,0.001887,5,39,10,0.923566,0.986128
1,42,0.625674,0.377261,0.376106,0.719214,0.001886,498,0.017980,12,11,130,0.881132,0.622046
2,1874,0.551362,0.525192,0.525764,0.604171,0.022490,54,0.054381,6,113,10,0.788882,0.941931
3,902745,0.574861,0.458382,0.455529,0.713463,0.009619,118,0.001520,5,22,14,0.808176,0.597278
4,1,0.735319,0.697278,0.705825,1.024213,0.131601,218,0.099579,10,62,10,0.894714,0.999841
